# Fluxigut — QIIME2 Preprocessing Pipeline


## 1. Import & Demultiplex

In [ ]:
# Import paired-end sequences
qiime tools import \
  --type 'SampleData[PairedEndSequencesWithQuality]' \
  --input-path raw_data \
  --input-format CasavaOneEightSingleLanePerSampleDirFmt \
  --output-path Analysis/demux-paired-end.qza

# Summarize import for quality inspection
qiime demux summarize \
  --i-data Analysis/demux-paired-end.qza \
  --o-visualization Analysis/demux-paired-end.qzv

## 2. DADA2 Denoising

In [ ]:
cd Analysis
mkdir dada2

# Denoise paired-end reads
# Truncation at 240 bp both directions; min-fold-parent-over-abundance=4.0 for chimera filtering
qiime dada2 denoise-paired \
  --i-demultiplexed-seqs demux-paired-end.qza \
  --p-trunc-len-f 240 \
  --p-trunc-len-r 240 \
  --p-min-fold-parent-over-abundance 4.0 \
  --p-n-threads 5 \
  --o-representative-sequences dada2/dada2-rep-seqs.qza \
  --o-table dada2/dada2-table.qza \
  --o-denoising-stats dada2/dada2-stats.qza

cd dada2

# Visualize denoising stats, feature table, and representative sequences
qiime metadata tabulate \
  --m-input-file dada2-stats.qza \
  --o-visualization dada2-stats.qzv

qiime feature-table summarize \
  --i-table dada2-table.qza \
  --o-visualization dada2-table.qzv

qiime feature-table tabulate-seqs \
  --i-data dada2-rep-seqs.qza \
  --o-visualization dada2-rep-seqs.qzv

## 3. Decontamination

Contaminants identified using the prevalence method, comparing samples against blank controls (`IsBlank == TRUE`).
Features with decontam score p ≤ 0.1 are treated as contaminants and removed.

In [ ]:
mkdir decontam

# Identify contaminants by prevalence method using blank controls
qiime quality-control decontam-identify \
  --i-table dada2-table.qza \
  --m-metadata-file LP8_metadata.tsv \
  --p-method prevalence \
  --p-prev-control-column IsBlank \
  --p-prev-control-indicator TRUE \
  --o-decontam-scores decontam/scores_global.qza

# Visualize decontam scores
qiime quality-control decontam-score-viz \
  --i-decontam-scores decontam/scores_global.qza \
  --i-table dada2-table.qza \
  --o-visualization decontam/merged_scores.qzv

In [ ]:
# Extract contaminant feature IDs (p <= 0.1)
qiime feature-table filter-features \
  --i-table dada2-table.qza \
  --m-metadata-file decontam/scores_global.qza \
  --p-where "[p] <= 0.1" \
  --o-filtered-table decontam/global_contam_only.qza

# Export contaminant table and extract IDs as plain text
# Note: BIOM-exported TSVs have two header lines; tail -n +3 skips both
qiime tools export \
  --input-path decontam/global_contam_only.qza \
  --output-path decontam/global_export

biom convert \
  -i decontam/global_export/feature-table.biom \
  -o decontam/global_export/global_contam.tsv \
  --to-tsv

tail -n +3 decontam/global_export/global_contam.tsv | cut -f1 > decontam/global_export/global_contaminants.txt

# Remove contaminants from feature table and representative sequences
qiime feature-table filter-features \
  --i-table dada2-table.qza \
  --m-metadata-file decontam/global_export/global_contaminants.txt \
  --p-exclude-ids \
  --o-filtered-table dada2-table_clean.qza

qiime feature-table filter-seqs \
  --i-data dada2-rep-seqs.qza \
  --m-metadata-file decontam/global_export/global_contaminants.txt \
  --p-exclude-ids \
  --o-filtered-data dada2-rep-seqs_clean.qza

# Verify cleaned table
qiime feature-table summarize \
  --i-table dada2-table_clean.qza \
  --o-visualization dada2-table_clean.qzv

## 4. Taxonomic Classification

Taxonomy assigned using the SILVA 138 99% NB classifier, human-stool weighted.

In [ ]:
mkdir Results

# Classify with SILVA 138 weighted classifier
qiime feature-classifier classify-sklearn \
  --i-classifier /scratch/jpenadiaz/Classifiers/silva-138-99-nb-human-stool-weighted-classifier.qza \
  --i-reads dada2-rep-seqs_clean.qza \
  --o-classification Results/Silva_weighted_taxonomy.qza

# Taxonomy barplot
qiime taxa barplot \
  --i-table dada2-table_clean.qza \
  --i-taxonomy Results/Silva_weighted_taxonomy.qza \
  --m-metadata-file LP8_metadata.tsv \
  --o-visualization Results/Silva_weighted_taxa-bar-plots.qzv

## 5. Export

Export taxonomy and feature table for downstream analysis in R.

In [ ]:
# Export SILVA weighted taxonomy
qiime tools export \
  --input-path Results/Silva_weighted_taxonomy.qza \
  --output-path Results/Silva_weighted_taxonomy_exported

# Export cleaned feature table (BIOM + TSV)
qiime tools export \
  --input-path dada2-table_clean.qza \
  --output-path Results/table_exported

biom convert \
  -i Results/table_exported/feature-table.biom \
  -o Results/table_exported/feature-table.tsv \
  --to-tsv